In [95]:
import pandas as pd
y_hat = pd.read_csv('./acc_test/thunder_hat_LOW.csv')
y_gt = pd.read_csv('./acc_test/keypoints_LOW.csv')
y = pd.read_csv('./acc_test/thunder_y_LOW.csv')
X = pd.read_csv('./acc_test/thunder_LOW.csv')
z_limit = 20
tor = 6

In [96]:
y_gt['X'], y_gt['Y'] = y_gt['Y'], y_gt['X']
y_gt[['X']] += 6 * 128 / 2 + 0.5
y_gt[['Y']] += 6 * 128 / 2 + 0.5

In [97]:
y_gt[['Image_Index']] += 1
y_gt[['Points_Index']] += 1
y_gt = y_gt[abs(y_gt['Z']) <= z_limit]

In [112]:
max(y_gt['Y'])

768.5

In [98]:
# 按 'frame' 进行匹配
def match(df1, df2, range):
    def match_within_frame(df1_group):
        frame_num = df1_group["Image_Index"].iloc[0]
        df2_group = df2[df2["frame"] == frame_num]  # 取相同帧的 df2 数据
        df1_group["matched"] = df1_group.apply(
            lambda row: df2_group[
                #(df2_group["x [nm]"] >= row['X'] - range) & (df2_group["x [nm]"] <= row['X'] + range) &  # x 范围
                #(df2_group["y [nm]"] >= row['Y'] - range) & (df2_group["y [nm]"] <= row['Y'] + range)     # y 范围
                (df2_group["x [nm]"]-row['X'])**2+(df2_group["y [nm]"]-row['Y'])**2 <= range**2
            ][["x [nm]", "y [nm]"]].values.tolist(), axis=1
        )
        return df1_group
    y_gt_matched = df1.groupby("Image_Index").apply(match_within_frame).reset_index(drop=True)
    return y_gt_matched
X_matched = match(y_gt, X, tor)
y_hat_matched = match(y_gt, y_hat, tor)
y_matched = match(y_gt, y, tor)

/var/folders/0r/w4xk_f1s0yjbn38ffy83cvj00000gn/T/ipykernel_1418/2300924604.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  y_gt_matched = df1.groupby("Image_Index").apply(match_within_frame).reset_index(drop=True)
/var/folders/0r/w4xk_f1s0yjbn38ffy83cvj00000gn/T/ipykernel_1418/2300924604.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  y_gt_matched = df1.groupby("Image_Index").apply(match_within_fr

In [99]:
def count_match_ratio(df):
    counter = 0
    points = []
    for i in range(len(df)):
        if len(df['matched'][i]) == 0:
            counter += 1
        else:
            points.append(i)
    rate = (len(df)- counter)/len(df)
    print(rate)
    # dropped = df.drop(index = points)
    return points
X_matched_points = count_match_ratio(X_matched)
y_hat_matched_points = count_match_ratio(y_hat_matched)
y_matched_points = count_match_ratio(y_matched)
print (len(X_matched_points)/len(y_matched_points))
print (len(y_hat_matched_points)/len(y_matched_points))

0.867252318204002
0.8989751098096632
0.9038555392874573
0.9595032397408207
0.9946004319654428


In [102]:
X_matched_points_filtered = [i for i in X_matched_points if i in y_matched_points]
y_hat_matched_points_filtered = [i for i in y_hat_matched_points if i in y_matched_points]
print (len(X_matched_points_filtered)/len(y_matched_points))
print (len(y_hat_matched_points_filtered)/len(y_matched_points))

0.953023758099352
0.9919006479481641


In [103]:
import numpy as np
def compute_acc_with_y(X_df, X_filt):
    acc_count = []
    for item in y_matched_points:
        if (item in X_filt) and (len(X_df['matched'].iloc[item]) == 1):
            acc_count.append(((X_df['matched'].iloc[item][0][0] - X_df['X'].iloc[item])**2 + (X_df['matched'].iloc[item][0][1] - X_df['Y'].iloc[item])**2) ** 0.5)
    acc = sum(acc_count) / len(acc_count)
    print(acc)
    return acc
acc_X = compute_acc_with_y(X_matched, X_matched_points_filtered)
acc_y_hat = compute_acc_with_y(y_hat_matched, y_hat_matched_points_filtered)
acc_hat = compute_acc_with_y(y_matched, y_hat_matched_points)

1.9709432401518203
1.3094196358797945
0.1855110043491721


In [104]:
y_gt.shape

(2049, 6)